# Baseline : deux régressions logistiques (Morpion)

Ce notebook charge le dataset binaire, entraîne **deux** modèles `LogisticRegression` (cibles `x_wins` et `is_draw`), puis compare leurs métriques sur le jeu de test.

In [1]:
# ---------------------------------------------------------------------------
# Imports : pandas pour les données, sklearn pour ML et métriques
# ---------------------------------------------------------------------------
from pathlib import Path
from typing import Tuple

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split

In [2]:
# ---------------------------------------------------------------------------
# Constantes : chemin du CSV et noms des colonnes
# ---------------------------------------------------------------------------
DATA_PATH = Path("ressources") / "dataset.csv"

# 9 cases × 2 indicateurs (X et O) = 18 features binaires (ordre c0_x, c0_o, c1_x, …)
FEATURE_COLS = [f"c{i}_{s}" for i in range(9) for s in ("x", "o")]
TARGET_X_WINS = "x_wins"
TARGET_IS_DRAW = "is_draw"

In [3]:
def load_dataset(csv_path: Path) -> pd.DataFrame:
    """Charge le fichier CSV et retourne un DataFrame pandas."""
    return pd.read_csv(csv_path)


def split_features_and_targets(
    df: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.Series, pd.Series]:
    """
    Sépare les features X (18 colonnes binaires) et les deux cibles :
    - y_xwins : x_wins
    - y_draw  : is_draw
    """
    X = df[FEATURE_COLS]
    y_xwins = df[TARGET_X_WINS]
    y_draw = df[TARGET_IS_DRAW]
    return X, y_xwins, y_draw

In [4]:
def train_logistic_regression(
    X_train, y_train, *, random_state: int = 42
) -> LogisticRegression:
    """
    Entraîne une régression logistique (baseline sklearn).
    max_iter élevé pour éviter les avertissements de non-convergence sur certaines données.
    """
    model = LogisticRegression(max_iter=1000, random_state=random_state)
    model.fit(X_train, y_train)
    return model


def evaluate_binary_classifier(y_true, y_pred, model_name: str) -> None:
    """
    Affiche accuracy, F1, matrice de confusion et rapport de classification
    pour un problème de classification binaire.
    """
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    print("=" * 60)
    print(f"Modèle : {model_name}")
    print("=" * 60)
    print(f"Accuracy     : {acc:.4f}")
    print(f"F1-score     : {f1:.4f}")
    print("\nMatrice de confusion (lignes = vrai, colonnes = prédit) :")
    print(cm)
    print("\nClassification report :")
    print(classification_report(y_true, y_pred, zero_division=0))

In [5]:
# ---------------------------------------------------------------------------
# 1) Chargement du dataset
# ---------------------------------------------------------------------------
df = load_dataset(DATA_PATH)
print(f"Forme du dataset : {df.shape[0]} lignes, {df.shape[1]} colonnes")
df.head()

Forme du dataset : 10 lignes, 20 colonnes


,c0_x,c0_o,c1_x,c1_o,c2_x,c2_o,c3_x,c3_o,c4_x,c4_o,c5_x,c5_o,c6_x,c6_o,c7_x,c7_o,c8_x,c8_o,x_wins,is_draw
0,1,0,1,0,1,0,0,1,0,1,0,1,0,1,0,1,0,1,1,0
1,0,1,0,1,0,1,1,0,1,0,1,0,1,0,1,0,1,0,0,1
2,1,0,0,1,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,0
3,0,1,1,0,1,0,1,0,0,1,1,0,1,0,1,0,1,0,0,0
4,1,0,1,0,0,1,0,1,1,0,1,0,0,1,0,1,0,1,1,0


In [7]:
# ---------------------------------------------------------------------------
# 2) Séparation features / cibles
# ---------------------------------------------------------------------------
X, y_xwins, y_draw = split_features_and_targets(df)

In [8]:
# ---------------------------------------------------------------------------
# 3) Train / test split 80 % / 20 %
#    Un seul appel : X, y_xwins et y_draw sont découpés de façon alignée.
#    stratify=y_xwins : garde la proportion de classes sur x_wins (retirer si trop peu de données).
# ---------------------------------------------------------------------------
RANDOM_STATE = 42
TEST_SIZE = 0.2

(
    X_train,
    X_test,
    y_xwins_train,
    y_xwins_test,
    y_draw_train,
    y_draw_test,
) = train_test_split(
    X,
    y_xwins,
    y_draw,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_xwins,
)

print(f"Train : {len(X_train)} échantillons | Test : {len(X_test)} échantillons")

Train : 8 échantillons | Test : 2 échantillons


In [9]:
# ---------------------------------------------------------------------------
# 4) Entraînement : deux LogisticRegression indépendantes
# ---------------------------------------------------------------------------
model_xwins = train_logistic_regression(X_train, y_xwins_train, random_state=RANDOM_STATE)
model_draw = train_logistic_regression(X_train, y_draw_train, random_state=RANDOM_STATE)

In [10]:
# ---------------------------------------------------------------------------
# 5) Prédictions sur le jeu de test
# ---------------------------------------------------------------------------
y_xwins_pred = model_xwins.predict(X_test)
y_draw_pred = model_draw.predict(X_test)

In [14]:
# ---------------------------------------------------------------------------
# 6) Évaluation et comparaison des deux modèles
# ---------------------------------------------------------------------------
evaluate_binary_classifier(y_xwins_test, y_xwins_pred, "Prédiction de x_wins (X gagne)")
print()
evaluate_binary_classifier(y_draw_test, y_draw_pred, "Prédiction de is_draw (match nul)")

Modèle : Prédiction de x_wins (X gagne)
Accuracy     : 1.0000
F1-score     : 1.0000

Matrice de confusion (lignes = vrai, colonnes = prédit) :
[[1 0]
 [0 1]]

Classification report :
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2


Modèle : Prédiction de is_draw (match nul)
Accuracy     : 1.0000
F1-score     : 0.0000

Matrice de confusion (lignes = vrai, colonnes = prédit) :
[[2]]

Classification report :
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2



/home/dev/examen/ml-examen/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


In [15]:
# ---------------------------------------------------------------------------
# Résumé comparatif (tableau)
# ---------------------------------------------------------------------------
summary = pd.DataFrame(
    {
        "Modèle": ["x_wins", "is_draw"],
        "Accuracy": [
            accuracy_score(y_xwins_test, y_xwins_pred),
            accuracy_score(y_draw_test, y_draw_pred),
        ],
        "F1-score": [
            f1_score(y_xwins_test, y_xwins_pred, zero_division=0),
            f1_score(y_draw_test, y_draw_pred, zero_division=0),
        ],
    }
)
summary

,Modèle,Accuracy,F1-score
0,x_wins,1.0,1.0
1,is_draw,1.0,0.0
